In [1]:
import random
import torch

def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

class LinearRegression:
    
    def __init__(self, input_dim):
        self.w = torch.normal(0, 0.01, size=(input_dim, 1), requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)

    def forward(self, X):
        return torch.matmul(X, self.w) + self.b

    def parameters(self):
        return [self.w, self.b]

    def squared_loss(self, y_hat, y):
        return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

    def sgd(self, params, lr, batch_size):
        with torch.no_grad():
            for param in params:
                param -= lr * param.grad / batch_size
                param.grad.zero_()

    def train_epoch(self, X_batch, y_batch, lr, batch_size):
        y_hat = self.forward(X_batch)
        loss = self.squared_loss(y_hat, y_batch).sum()
        loss.backward()
        self.sgd(self.parameters(), lr, batch_size)
        return loss.item()

    def evaluate_loss(self, features, labels):
        with torch.no_grad():
            y_hat = self.forward(features)
            loss = self.squared_loss(y_hat, labels).sum().item()
        return loss

model = LinearRegression(input_dim=2)   
lr = 0.03
num_epochs = 3
batch_size = 10

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    for X_batch, y_batch in data_iter(batch_size, features, labels):
        loss = model.train_epoch(X_batch, y_batch, lr, batch_size)
        epoch_loss += loss
        num_batches += 1
    avg_loss = model.evaluate_loss(features, labels)
    print(f'epoch {epoch+1}, loss {avg_loss:.6f}')

print(f"w的误差: {true_w - model.w.reshape(true_w.shape)}")
print(f"b的误差: {true_b - model.b}")

epoch 1, loss 31.309792
epoch 2, loss 0.115257
epoch 3, loss 0.052521
w的误差: tensor([-5.3453e-04, -3.4809e-05], grad_fn=<SubBackward0>)
b的误差: tensor([3.8147e-05], grad_fn=<RsubBackward1>)
